In [19]:
# ============================================================
# INGEST CIRCUITS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import *
import nbformat
from nbconvert import PythonExporter

In [20]:
# -----------------------------------------
# Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [21]:
# -----------------------------------------
# Detect latest batch folder
# -----------------------------------------
batch_folders = sorted([
    f for f in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, f))
])

if not batch_folders:
    raise Exception("❌ No batch folders found in landing!")

p_batch_id = batch_folders[-1]
print("Detected batch folder:", p_batch_id)

BATCH_LANDING_PATH = f"{LANDING_PATH}/{p_batch_id}"

Detected batch folder: 2025-01


In [22]:
# -----------------------------------------
# Generate batch_id for Bronze write
# -----------------------------------------
batch_id = generate_batch_id()
print("Pipeline batch_id:", batch_id)

Pipeline batch_id: 20260609_130719


In [23]:
# -----------------------------------------
# Define source + target paths
# -----------------------------------------
source_file = f"{BATCH_LANDING_PATH}/circuits.csv"
target_path = f"{BRONZE_PATH}/circuits"
os.makedirs(target_path, exist_ok=True)


In [24]:
# -----------------------------------------
# Schema
# -----------------------------------------
circuits_schema = StructType([
    StructField('circuitId', StringType(), True),
    StructField('url', StringType(), True),
    StructField('circuitName', StringType(), True),
    StructField('lat', DoubleType(), True),
    StructField('long', DoubleType(), True),
    StructField('locality', StringType(), True),
    StructField('country', StringType(), True)
])

In [25]:
# -----------------------------------------
# Read
# -----------------------------------------
df = (
    spark.read
        .format("csv")
        .schema(circuits_schema)
        .option("header", True)
        .load(source_file)
)

In [26]:
# -----------------------------------------
# Metadata
# -----------------------------------------
df_final = add_ingestion_metadata(df, source_file)

In [27]:
# -----------------------------------------
# Write Bronze
# -----------------------------------------
write_to_bronze(df_final, f"{target_path}/data.parquet", batch_id)

In [28]:
# -----------------------------------------
# Validate
# -----------------------------------------
spark.read.parquet(f"{target_path}/data.parquet").show(5)

+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+---------------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country|    ingest_timestamp|         source_file|       batch_id|
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+---------------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|2026-06-09 13:07:...|/Users/manoharaza...|20260609_130719|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|casablanca|  Morocco|20